<style>
.lab-banner {
  padding: 28px 32px;
  border-radius: 18px;
  background: linear-gradient(120deg, #0f172a, #0f766e, #2563eb);
  color: white;
  margin: 8px 0 22px 0;
  box-shadow: 0 10px 30px rgba(15, 23, 42, .22);
}
.lab-banner h1 { margin: 0 0 8px 0; font-size: 34px; }
.lab-banner p { margin: 5px 0; font-size: 16px; opacity: .96; }
.note-box, .why-box, .warning-box, .result-box, .exercise-box {
  padding: 14px 18px; border-radius: 12px; margin: 12px 0;
}
.note-box { background: #eff6ff; border-left: 5px solid #2563eb; }
.why-box { background: #ecfdf5; border-left: 5px solid #059669; }
.warning-box { background: #fff7ed; border-left: 5px solid #ea580c; }
.result-box { background: #f5f3ff; border-left: 5px solid #7c3aed; }
.exercise-box { background: #fefce8; border-left: 5px solid #ca8a04; }
.small { color: #475569; font-size: 13px; }
</style>

<div class="lab-banner">
  <h1>LAB 09 — T5, BART và ViT5</h1>
  <p><b>Tóm tắt văn bản bằng Transformer Encoder–Decoder</b></p>
  <p>Hugging Face · PyTorch · Seq2SeqTrainer · ROUGE · Google Colab GPU</p>
</div>

<div class="note-box">
<b>Tiếp nối LAB 08:</b> BERT/PhoBERT chủ yếu mã hóa văn bản để phân loại. LAB 09 chuyển sang kiến trúc <b>sequence-to-sequence</b>, trong đó encoder đọc văn bản và decoder sinh bản tóm tắt mới.
</div>

## Mục tiêu học tập

Sau khi hoàn thành notebook, người học có thể:

1. Phân biệt tóm tắt **extractive** và **abstractive**.
2. Giải thích kiến trúc Transformer encoder–decoder và cơ chế teacher forcing.
3. Phân biệt T5, BART và ViT5.
4. Chuẩn bị dữ liệu nguồn–đích cho bài toán sequence-to-sequence.
5. Fine-tune T5 và BART trên dữ liệu hội thoại tiếng Anh SAMSum.
6. Fine-tune ViT5 trên dữ liệu tin tức tiếng Việt VietNews.
7. Đánh giá bằng ROUGE-1, ROUGE-2 và ROUGE-L.
8. So sánh greedy search, beam search và nucleus sampling.
9. Phân tích các bản tóm tắt yếu và nhận diện nguy cơ hallucination.
10. Lưu, tải lại và dùng mô hình để tóm tắt văn bản mới.

## Sản phẩm cuối lab

Notebook tạo ra:

- Một mô hình **T5-small** fine-tune cho tóm tắt hội thoại tiếng Anh.
- Một mô hình **BART-base** fine-tune trên cùng dữ liệu để so sánh công bằng.
- Một mô hình **ViT5-base** fine-tune cho tóm tắt tin tức tiếng Việt.
- Bảng benchmark chung: ROUGE, thời gian, số tham số và peak VRAM.
- Bảng phân tích các dự đoán có ROUGE-L thấp.
- Demo ba chiến lược sinh văn bản.
- Thư mục mô hình trong `/content/lab09_models/`.

<div class="warning-box"><b>Chi phí tính toán:</b> Fine-tune cả ba mô hình cần GPU. Chạy lần đầu với <code>QUICK_MODE = True</code>; chỉ bật chế độ đầy đủ sau khi toàn bộ pipeline đã chạy thành công.</div>

# PHẦN A — LÝ THUYẾT NỀN

## 1. Tóm tắt extractive và abstractive

| Loại | Cách hoạt động | Ưu điểm | Hạn chế |
|---|---|---|---|
| Extractive | Chọn các câu hoặc cụm từ có sẵn | Ít bịa thông tin, dễ giải thích | Có thể dài, thiếu tự nhiên |
| Abstractive | Sinh câu mới biểu đạt nội dung chính | Ngắn gọn, tự nhiên | Có thể hallucinate hoặc bỏ sót |

LAB này tập trung vào **abstractive summarization**.

## 2. Kiến trúc encoder–decoder

```text
Văn bản nguồn
    ↓ tokenizer
[input_ids, attention_mask]
    ↓
Transformer Encoder
    ↓ biểu diễn ngữ cảnh
Transformer Decoder
    ↓ sinh tự hồi quy
Bản tóm tắt
```

Khi huấn luyện, decoder nhận các token đúng trước đó của bản tóm tắt. Khi suy luận, decoder phải dùng chính token nó vừa sinh để tạo token tiếp theo.

Với chuỗi nguồn $x$ và chuỗi đích $y$:

$$
\mathcal{L}=-\sum_{t=1}^{T}\log P(y_t\mid y_{<t},x)
$$

Đây là Cross-Entropy trên từng vị trí token của chuỗi đích.

## 3. T5, BART và ViT5

| Mô hình | Ý tưởng chính | Ngôn ngữ | Ghi chú trong lab |
|---|---|---|---|
| T5 | Biểu diễn mọi nhiệm vụ dưới dạng text-to-text | Chủ yếu tiếng Anh | Dùng tiền tố `summarize:` |
| BART | Encoder hai chiều + decoder tự hồi quy; học khôi phục văn bản bị làm nhiễu | Chủ yếu tiếng Anh | Không cần tiền tố nhiệm vụ |
| ViT5 | T5 được tiền huấn luyện chuyên biệt cho tiếng Việt | Tiếng Việt | Fine-tune trên bài báo → abstract |

Số tham số được notebook tính trực tiếp sau khi tải model để tránh phụ thuộc vào con số ghi nhớ thủ công.

## 4. ROUGE đánh giá điều gì?

- **ROUGE-1:** mức trùng khớp unigram.
- **ROUGE-2:** mức trùng khớp bigram.
- **ROUGE-L:** dựa trên dãy con chung dài nhất.

ROUGE hữu ích để benchmark tự động, nhưng không đủ để xác nhận bản tóm tắt đúng sự thật. Vì vậy lab còn yêu cầu đọc mẫu, kiểm tra thông tin bị bỏ sót và hallucination.

## 5. Chiến lược giải mã

| Chiến lược | Thiết lập | Đặc điểm |
|---|---|---|
| Greedy | Chọn token xác suất cao nhất ở mỗi bước | Nhanh, đôi khi kém tối ưu toàn cục |
| Beam search | Giữ nhiều chuỗi ứng viên | Phổ biến cho tóm tắt, ổn định hơn |
| Nucleus sampling | Lấy mẫu trong tập token có tổng xác suất `top_p` | Đa dạng nhưng dễ thêm thông tin không có trong nguồn |

<div class="why-box"><b>Khuyến nghị:</b> Beam search thường phù hợp hơn sampling cho bài toán cần tính trung thực như tóm tắt tin tức.</div>

# PHẦN B — CHUẨN BỊ GOOGLE COLAB

## 6. Hướng dẫn chạy

1. Chọn **Runtime → Change runtime type → T4 GPU** hoặc GPU mạnh hơn.
2. Chạy notebook từ trên xuống bằng **Runtime → Run all**.
3. Giữ `QUICK_MODE = True` trong lần chạy đầu.
4. Không import TensorFlow; lab dùng PyTorch nên tránh chuỗi lỗi TensorFlow/Protobuf của notebook trước.
5. Dữ liệu và model được tải từ Hugging Face Hub, vì vậy cần kết nối Internet.

In [ ]:
# Cài thư viện trước mọi import Hugging Face.
# Không nâng cấp PyTorch của Colab để tránh làm mất bản CUDA tương thích sẵn có.
import subprocess
import sys

packages = [
    "transformers>=4.48,<6",
    "datasets>=3.2,<6",
    "accelerate>=1.2,<2",
    "evaluate>=0.4.3,<1",
    "rouge_score>=0.1.2",
    "sentencepiece>=0.2.0",
    "safetensors>=0.4.5",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "--upgrade-strategy", "only-if-needed", *packages
])
print("Cài thư viện hoàn tất.")

In [ ]:
import gc
import inspect
import json
import os
import platform
import random
import re
import shutil
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
import datasets

from datasets import Dataset, DatasetDict, load_dataset
from IPython.display import display
from rouge_score import rouge_scorer
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

try:
    import evaluate
except Exception:
    evaluate = None

warnings.filterwarnings("ignore", category=FutureWarning)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA runtime:", torch.version.cuda)

In [ ]:
# =========================
# CẤU HÌNH THÍ NGHIỆM
# =========================
SEED = 42
QUICK_MODE = True

RUN_T5 = True
RUN_BART = True
RUN_VIT5 = True
RUN_GRADIO = False  # Chỉ bật sau khi các mô hình đã huấn luyện xong.

MODEL_IDS = {
    "T5-small": "google-t5/t5-small",
    "BART-base": "facebook/bart-base",
    "ViT5-base": "VietAI/vit5-base",
}

if QUICK_MODE:
    EN_TRAIN_SIZE, EN_VAL_SIZE, EN_TEST_SIZE = 1_200, 160, 200
    VI_TRAIN_SIZE, VI_VAL_SIZE, VI_TEST_SIZE = 1_200, 160, 200
    EN_SOURCE_MAX, EN_TARGET_MAX = 256, 80
    VI_SOURCE_MAX, VI_TARGET_MAX = 384, 96
    EPOCHS = 1
else:
    EN_TRAIN_SIZE, EN_VAL_SIZE, EN_TEST_SIZE = 6_000, 500, 500
    VI_TRAIN_SIZE, VI_VAL_SIZE, VI_TEST_SIZE = 6_000, 500, 500
    EN_SOURCE_MAX, EN_TARGET_MAX = 384, 96
    VI_SOURCE_MAX, VI_TARGET_MAX = 512, 128
    EPOCHS = 2

EXPERIMENT_CONFIGS = {
    "T5-small": {
        "train_batch": 8,
        "eval_batch": 16,
        "grad_accum": 2,
        "learning_rate": 5e-5,
        "prefix": "summarize: ",
    },
    "BART-base": {
        "train_batch": 4,
        "eval_batch": 8,
        "grad_accum": 4,
        "learning_rate": 3e-5,
        "prefix": "",
    },
    "ViT5-base": {
        "train_batch": 2,
        "eval_batch": 4,
        "grad_accum": 8,
        "learning_rate": 3e-5,
        "prefix": "",
    },
}

OUTPUT_ROOT = Path("/content/lab09_models")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

GPU_AVAILABLE = torch.cuda.is_available()
if not GPU_AVAILABLE:
    raise RuntimeError(
        "Không phát hiện GPU. Hãy chọn Runtime → Change runtime type → T4 GPU rồi chạy lại từ đầu."
    )

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("QUICK_MODE:", QUICK_MODE)
print("Epochs:", EPOCHS)
print("Output:", OUTPUT_ROOT)

<div class="note-box"><b>Tối ưu GPU:</b> FP16, gradient checkpointing, dynamic padding, fused AdamW, TF32, ghim bộ nhớ DataLoader và gradient accumulation. Mỗi mô hình được giải phóng khỏi GPU sau khi lưu để tránh tràn VRAM khi chạy tuần tự.</div>

In [ ]:
props = torch.cuda.get_device_properties(0)
print(f"Tên GPU: {props.name}")
print(f"VRAM tổng: {props.total_memory / 1024**3:.2f} GB")
print(f"Compute capability: {props.major}.{props.minor}")
print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")

# PHẦN C — TẢI VÀ KHẢO SÁT DỮ LIỆU

## 7. SAMSum: hội thoại tiếng Anh

Các trường chính:

- `dialogue`: hội thoại nguồn.
- `summary`: bản tóm tắt do con người viết.
- `id`: mã mẫu.

T5 và BART dùng **cùng split và cùng kích thước mẫu** để việc so sánh có ý nghĩa hơn.

In [ ]:
def iterable_to_dataset(iterable, size, seed=SEED, buffer_size=10_000):
    """Lấy tập con từ streaming dataset để giảm lượng dữ liệu phải tải."""
    stream = iterable.shuffle(seed=seed, buffer_size=buffer_size)
    rows = list(stream.take(size))
    if not rows:
        raise RuntimeError("Không lấy được dữ liệu từ Hugging Face Hub.")
    return Dataset.from_list(rows)


def load_streaming_subset(repo_id, split, size, seed=SEED):
    stream = load_dataset(repo_id, split=split, streaming=True)
    return iterable_to_dataset(stream, size=size, seed=seed)


# SAMSum dùng tên split validation trên Hub.
english_raw = DatasetDict({
    "train": load_streaming_subset("knkarthick/samsum", "train", EN_TRAIN_SIZE, SEED),
    "validation": load_streaming_subset("knkarthick/samsum", "validation", EN_VAL_SIZE, SEED + 1),
    "test": load_streaming_subset("knkarthick/samsum", "test", EN_TEST_SIZE, SEED + 2),
})

english_raw

In [ ]:
print(english_raw["train"].features)
example_en = english_raw["train"][0]
print("HỘI THOẠI:\n", example_en["dialogue"])
print("\nTÓM TẮT THAM CHIẾU:\n", example_en["summary"])

## 8. VietNews: tóm tắt tin tức tiếng Việt

Các trường chính:

- `article`: nội dung bài báo.
- `abstract`: tóm tắt tham chiếu.
- `title`: tiêu đề.
- `guid`: mã mẫu.

Dữ liệu có thể chứa dấu gạch dưới nối các âm tiết trong từ ghép. Lab giữ nguyên định dạng của nguồn khi fine-tune để nhất quán với dữ liệu VietNews.

In [ ]:
# Streaming đặc biệt hữu ích vì kho VietNews lớn hơn đáng kể so với tập con cần dùng.
vietnamese_raw = DatasetDict({
    "train": load_streaming_subset("ithieund/VietNews-Abs-Sum", "train", VI_TRAIN_SIZE, SEED),
    "validation": load_streaming_subset("ithieund/VietNews-Abs-Sum", "validation", VI_VAL_SIZE, SEED + 1),
    "test": load_streaming_subset("ithieund/VietNews-Abs-Sum", "test", VI_TEST_SIZE, SEED + 2),
})

vietnamese_raw

In [ ]:
print(vietnamese_raw["train"].features)
example_vi = vietnamese_raw["train"][0]
print("TIÊU ĐỀ:\n", example_vi.get("title", ""))
print("\nBÀI BÁO:\n", example_vi["article"][:1500])
print("\nTÓM TẮT THAM CHIẾU:\n", example_vi["abstract"])

In [ ]:
def basic_length_stats(dataset_dict, source_col, target_col, name):
    rows = []
    for split, ds in dataset_dict.items():
        src_len = [len(str(x).split()) for x in ds[source_col]]
        tgt_len = [len(str(x).split()) for x in ds[target_col]]
        rows.append({
            "dataset": name,
            "split": split,
            "n": len(ds),
            "source_mean_words": round(float(np.mean(src_len)), 1),
            "source_p95_words": round(float(np.percentile(src_len, 95)), 1),
            "target_mean_words": round(float(np.mean(tgt_len)), 1),
            "target_p95_words": round(float(np.percentile(tgt_len, 95)), 1),
        })
    return pd.DataFrame(rows)

length_stats = pd.concat([
    basic_length_stats(english_raw, "dialogue", "summary", "SAMSum"),
    basic_length_stats(vietnamese_raw, "article", "abstract", "VietNews"),
], ignore_index=True)
display(length_stats)

<div class="why-box"><b>Vì sao phải giới hạn chiều dài?</b> Self-attention tiêu tốn bộ nhớ theo bình phương chiều dài chuỗi. Cắt chuỗi hợp lý giúp giảm VRAM và tăng tốc, nhưng cắt quá ngắn có thể làm mất thông tin quan trọng ở cuối tài liệu.</div>

# PHẦN D — HÀM DÙNG CHUNG CHO SEQ2SEQ

In [ ]:
# Cố gắng dùng Hugging Face Evaluate; nếu Hub metric không tải được thì dùng rouge_score cục bộ.
try:
    HF_ROUGE = evaluate.load("rouge") if evaluate is not None else None
    print("ROUGE backend: Hugging Face Evaluate")
except Exception as exc:
    HF_ROUGE = None
    print("Không tải được evaluate/rouge; dùng fallback rouge_score.")
    print(type(exc).__name__, str(exc)[:200])


def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()


def compute_rouge_scores(predictions, references):
    predictions = [normalize_text(x) for x in predictions]
    references = [normalize_text(x) for x in references]

    if HF_ROUGE is not None:
        result = HF_ROUGE.compute(
            predictions=predictions,
            references=references,
            use_stemmer=False,
        )
        return {k: float(v) * 100 for k, v in result.items() if k in {"rouge1", "rouge2", "rougeL", "rougeLsum"}}

    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=False,
    )
    collected = {"rouge1": [], "rouge2": [], "rougeL": []}
    for pred, ref in zip(predictions, references):
        score = scorer.score(ref, pred)
        for key in collected:
            collected[key].append(score[key].fmeasure)
    return {key: float(np.mean(values)) * 100 for key, values in collected.items()}

In [ ]:
def make_preprocess_function(tokenizer, source_col, target_col, prefix, source_max, target_max):
    def preprocess(batch):
        sources = [prefix + normalize_text(x) for x in batch[source_col]]
        targets = [normalize_text(x) for x in batch[target_col]]

        model_inputs = tokenizer(
            sources,
            max_length=source_max,
            truncation=True,
        )
        labels = tokenizer(
            text_target=targets,
            max_length=target_max,
            truncation=True,
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
    return preprocess


def make_compute_metrics(tokenizer):
    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        if isinstance(predictions, tuple):
            predictions = predictions[0]

        predictions = np.asarray(predictions)
        labels = np.asarray(labels)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

        decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        scores = compute_rouge_scores(decoded_preds, decoded_labels)

        non_pad = predictions != tokenizer.pad_token_id
        scores["gen_len"] = float(np.mean(non_pad.sum(axis=1)))
        return {k: round(v, 4) for k, v in scores.items()}
    return compute_metrics

In [ ]:
def build_seq2seq_training_arguments(
    output_dir,
    train_batch,
    eval_batch,
    grad_accum,
    learning_rate,
    target_max,
):
    """Tạo TrainingArguments tương thích với nhiều phiên bản Transformers 4.x/5.x."""
    params = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
    bf16_ok = bool(GPU_AVAILABLE and torch.cuda.is_bf16_supported())

    kwargs = {
        "output_dir": str(output_dir),
        "learning_rate": learning_rate,
        "per_device_train_batch_size": train_batch,
        "per_device_eval_batch_size": eval_batch,
        "gradient_accumulation_steps": grad_accum,
        "num_train_epochs": EPOCHS,
        "weight_decay": 0.01,
        "warmup_ratio": 0.06,
        "logging_steps": 20,
        "save_total_limit": 1,
        "load_best_model_at_end": True,
        "metric_for_best_model": "rougeL",
        "greater_is_better": True,
        "report_to": "none",
        "seed": SEED,
        "data_seed": SEED,
        "dataloader_num_workers": min(2, os.cpu_count() or 1),
        "dataloader_pin_memory": True,
        "fp16": GPU_AVAILABLE and not bf16_ok,
        "bf16": bf16_ok,
        "tf32": GPU_AVAILABLE,
        "gradient_checkpointing": True,
        "optim": "adamw_torch_fused" if GPU_AVAILABLE else "adamw_torch",
        "predict_with_generate": True,
        "generation_max_length": target_max,
        "generation_num_beams": 4,
        "save_safetensors": True,
    }

    if "eval_strategy" in params:
        kwargs["eval_strategy"] = "epoch"
    elif "evaluation_strategy" in params:
        kwargs["evaluation_strategy"] = "epoch"

    kwargs["save_strategy"] = "epoch"
    kwargs["logging_strategy"] = "steps"

    safe_kwargs = {k: v for k, v in kwargs.items() if k in params}
    return Seq2SeqTrainingArguments(**safe_kwargs)


def build_seq2seq_trainer(model, args, tokenizer, data_collator, train_dataset, eval_dataset):
    trainer_kwargs = {
        "model": model,
        "args": args,
        "train_dataset": train_dataset,
        "eval_dataset": eval_dataset,
        "data_collator": data_collator,
        "compute_metrics": make_compute_metrics(tokenizer),
    }

    params = inspect.signature(Seq2SeqTrainer.__init__).parameters
    if "processing_class" in params:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in params:
        trainer_kwargs["tokenizer"] = tokenizer

    return Seq2SeqTrainer(**trainer_kwargs)

In [ ]:
def count_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def clean_metric_dict(metrics):
    cleaned = {}
    for key, value in metrics.items():
        key2 = key.replace("test_", "").replace("eval_", "")
        if isinstance(value, (int, float, np.number)):
            cleaned[key2] = float(value)
    return cleaned


def make_prediction_table(tokenizer, prediction_output, raw_test, source_col, target_col):
    pred_ids = prediction_output.predictions
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]
    decoded = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)

    rows = []
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    for i, pred in enumerate(decoded):
        source = normalize_text(raw_test[i][source_col])
        reference = normalize_text(raw_test[i][target_col])
        pred = normalize_text(pred)
        rouge_l = scorer.score(reference, pred)["rougeL"].fmeasure * 100
        rows.append({
            "source": source,
            "reference": reference,
            "prediction": pred,
            "rougeL": rouge_l,
        })
    return pd.DataFrame(rows)

In [ ]:
def load_tokenizer_robust(model_id):
    """Ưu tiên fast tokenizer; tự fallback sang slow tokenizer nếu checkpoint không hỗ trợ."""
    try:
        return AutoTokenizer.from_pretrained(model_id, use_fast=True)
    except Exception as exc:
        print("Fast tokenizer không khả dụng; chuyển sang slow tokenizer:", type(exc).__name__)
        return AutoTokenizer.from_pretrained(model_id, use_fast=False)


benchmark_rows = []
PREDICTION_TABLES = {}
SAVED_MODELS = {}


def run_seq2seq_experiment(
    experiment_name,
    model_id,
    raw_dataset,
    source_col,
    target_col,
    source_max,
    target_max,
):
    cfg = EXPERIMENT_CONFIGS[experiment_name]
    output_dir = OUTPUT_ROOT / experiment_name.lower().replace("-", "_")
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 80)
    print("THÍ NGHIỆM:", experiment_name)
    print("MODEL:", model_id)
    print("OUTPUT:", output_dir)

    clear_gpu()
    tokenizer = load_tokenizer_robust(model_id)
    preprocess_fn = make_preprocess_function(
        tokenizer=tokenizer,
        source_col=source_col,
        target_col=target_col,
        prefix=cfg["prefix"],
        source_max=source_max,
        target_max=target_max,
    )

    tokenized = raw_dataset.map(
        preprocess_fn,
        batched=True,
        remove_columns=raw_dataset["train"].column_names,
        desc=f"Tokenize {experiment_name}",
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    model.generation_config.max_length = target_max
    model.generation_config.num_beams = 4
    model.generation_config.no_repeat_ngram_size = 3
    model.generation_config.early_stopping = True

    trainable, total = count_parameters(model)
    print(f"Parameters: {total/1e6:.2f}M | trainable: {trainable/1e6:.2f}M")

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        label_pad_token_id=-100,
        pad_to_multiple_of=8,
    )

    args = build_seq2seq_training_arguments(
        output_dir=output_dir,
        train_batch=cfg["train_batch"],
        eval_batch=cfg["eval_batch"],
        grad_accum=cfg["grad_accum"],
        learning_rate=cfg["learning_rate"],
        target_max=target_max,
    )

    trainer = build_seq2seq_trainer(
        model=model,
        args=args,
        tokenizer=tokenizer,
        data_collator=data_collator,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
    )

    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    train_output = trainer.train()
    train_seconds = time.perf_counter() - start

    model.config.use_cache = True
    prediction_output = trainer.predict(
        tokenized["test"],
        max_length=target_max,
        num_beams=4,
    )
    test_metrics = clean_metric_dict(prediction_output.metrics)
    peak_vram_gb = torch.cuda.max_memory_allocated() / 1024**3

    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))

    # Sau khi đã lưu model tốt nhất vào thư mục gốc, xóa checkpoint trung gian để tiết kiệm ổ đĩa.
    for checkpoint_dir in output_dir.glob("checkpoint-*"):
        shutil.rmtree(checkpoint_dir, ignore_errors=True)

    pred_table = make_prediction_table(
        tokenizer,
        prediction_output,
        raw_dataset["test"],
        source_col,
        target_col,
    )
    PREDICTION_TABLES[experiment_name] = pred_table
    SAVED_MODELS[experiment_name] = str(output_dir)

    row = {
        "model": experiment_name,
        "base_checkpoint": model_id,
        "train_examples": len(raw_dataset["train"]),
        "epochs": EPOCHS,
        "parameters_M": round(total / 1e6, 2),
        "train_minutes": round(train_seconds / 60, 2),
        "peak_vram_GB": round(peak_vram_gb, 2),
        "rouge1": round(test_metrics.get("rouge1", np.nan), 3),
        "rouge2": round(test_metrics.get("rouge2", np.nan), 3),
        "rougeL": round(test_metrics.get("rougeL", np.nan), 3),
        "gen_len": round(test_metrics.get("gen_len", np.nan), 2),
        "path": str(output_dir),
    }
    benchmark_rows.append(row)

    print("Kết quả test:", json.dumps(row, ensure_ascii=False, indent=2))
    display(pred_table.head(3))

    del trainer, model, tokenizer, tokenized, data_collator
    clear_gpu()
    return row

# PHẦN E — FINE-TUNE T5-SMALL

T5 coi mọi nhiệm vụ là chuyển đổi văn bản sang văn bản. Vì vậy đầu vào được thêm tiền tố:

```text
summarize: <dialogue>
```

In [ ]:
if RUN_T5:
    t5_result = run_seq2seq_experiment(
        experiment_name="T5-small",
        model_id=MODEL_IDS["T5-small"],
        raw_dataset=english_raw,
        source_col="dialogue",
        target_col="summary",
        source_max=EN_SOURCE_MAX,
        target_max=EN_TARGET_MAX,
    )
else:
    print("Bỏ qua T5-small.")

# PHẦN F — FINE-TUNE BART-BASE

BART được tiền huấn luyện bằng cách làm nhiễu văn bản rồi học khôi phục. Trong lab, BART dùng cùng SAMSum split với T5 và không cần tiền tố `summarize:`.

In [ ]:
if RUN_BART:
    bart_result = run_seq2seq_experiment(
        experiment_name="BART-base",
        model_id=MODEL_IDS["BART-base"],
        raw_dataset=english_raw,
        source_col="dialogue",
        target_col="summary",
        source_max=EN_SOURCE_MAX,
        target_max=EN_TARGET_MAX,
    )
else:
    print("Bỏ qua BART-base.")

# PHẦN G — FINE-TUNE ViT5-BASE

ViT5 dùng kiến trúc text-to-text cho tiếng Việt. Đầu vào là `article`, đầu ra là `abstract`.

<div class="warning-box"><b>Lưu ý bộ nhớ:</b> ViT5-base lớn hơn T5-small. Notebook dùng batch nhỏ, gradient accumulation và gradient checkpointing để phù hợp với GPU T4.</div>

In [ ]:
if RUN_VIT5:
    vit5_result = run_seq2seq_experiment(
        experiment_name="ViT5-base",
        model_id=MODEL_IDS["ViT5-base"],
        raw_dataset=vietnamese_raw,
        source_col="article",
        target_col="abstract",
        source_max=VI_SOURCE_MAX,
        target_max=VI_TARGET_MAX,
    )
else:
    print("Bỏ qua ViT5-base.")

# PHẦN H — SO SÁNH KẾT QUẢ

In [ ]:
benchmark_df = pd.DataFrame(benchmark_rows)
if benchmark_df.empty:
    print("Chưa có mô hình nào được chạy.")
else:
    benchmark_df = benchmark_df.sort_values("rougeL", ascending=False).reset_index(drop=True)
    display(benchmark_df)
    benchmark_df.to_csv(OUTPUT_ROOT / "lab09_benchmark.csv", index=False)
    print("Đã lưu:", OUTPUT_ROOT / "lab09_benchmark.csv")

In [ ]:
if not benchmark_df.empty:
    ax = benchmark_df.plot(
        x="model",
        y=["rouge1", "rouge2", "rougeL"],
        kind="bar",
        figsize=(10, 5),
        rot=0,
        title="So sánh ROUGE trên test subset",
    )
    ax.set_ylabel("ROUGE F1 × 100")
    ax.set_xlabel("")
    plt.tight_layout()
    plt.show()

<div class="note-box"><b>Không so sánh trực tiếp tuyệt đối tiếng Anh với tiếng Việt:</b> T5/BART và ViT5 dùng hai dataset khác nhau, độ dài khác nhau và ngôn ngữ khác nhau. Bảng chung chủ yếu để quan sát chi phí tính toán; so sánh chất lượng công bằng nhất là T5 với BART trên SAMSum.</div>

# PHẦN I — PHÂN TÍCH LỖI

Các mẫu ROUGE-L thấp không phải lúc nào cũng sai: một bản tóm tắt có thể đúng nghĩa nhưng dùng từ khác tham chiếu. Vì vậy cần đọc cả nguồn, tham chiếu và dự đoán.

In [ ]:
def show_low_rouge_examples(model_name, n=5, source_chars=700):
    if model_name not in PREDICTION_TABLES:
        print(f"Không có bảng dự đoán của {model_name}.")
        return
    df = PREDICTION_TABLES[model_name].nsmallest(n, "rougeL").copy()
    df["source"] = df["source"].str.slice(0, source_chars)
    display(df[["rougeL", "source", "reference", "prediction"]])

for model_name in PREDICTION_TABLES:
    print("\n", "=" * 30, model_name, "=" * 30)
    show_low_rouge_examples(model_name, n=3)

### Checklist phân tích thủ công

Với từng mẫu yếu, đánh dấu:

- Bỏ sót chủ thể, hành động hoặc kết quả quan trọng?
- Thêm tên, số liệu, địa điểm không có trong nguồn?
- Lặp từ hoặc lặp cụm?
- Bản tóm tắt quá ngắn hoặc quá dài?
- Khác từ tham chiếu nhưng vẫn đúng nghĩa?
- Nguồn đã bị truncation trước khi đến model?

# PHẦN J — SO SÁNH CHIẾN LƯỢC SINH

In [ ]:
def load_saved_seq2seq(model_name):
    if model_name not in SAVED_MODELS:
        raise KeyError(f"Chưa có model đã lưu: {model_name}")
    path = SAVED_MODELS[model_name]
    tokenizer = AutoTokenizer.from_pretrained(path)
    model = AutoModelForSeq2SeqLM.from_pretrained(path).to("cuda")
    model.eval()
    return tokenizer, model


@torch.inference_mode()
def summarize_with_strategy(
    text,
    tokenizer,
    model,
    prefix="",
    source_max=384,
    strategy="beam",
    max_new_tokens=96,
):
    inputs = tokenizer(
        prefix + normalize_text(text),
        return_tensors="pt",
        truncation=True,
        max_length=source_max,
    ).to(model.device)

    common = {
        "max_new_tokens": max_new_tokens,
        "min_new_tokens": 8,
        "no_repeat_ngram_size": 3,
        "repetition_penalty": 1.08,
    }

    if strategy == "greedy":
        gen_kwargs = {**common, "num_beams": 1, "do_sample": False}
    elif strategy == "beam":
        gen_kwargs = {
            **common,
            "num_beams": 4,
            "do_sample": False,
            "length_penalty": 1.0,
            "early_stopping": True,
        }
    elif strategy == "sampling":
        gen_kwargs = {
            **common,
            "num_beams": 1,
            "do_sample": True,
            "top_p": 0.9,
            "temperature": 0.8,
        }
    else:
        raise ValueError("strategy phải là greedy, beam hoặc sampling")

    output_ids = model.generate(**inputs, **gen_kwargs)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [ ]:
# Chọn một model đã fine-tune để so sánh ba chiến lược.
DEMO_MODEL = "BART-base" if "BART-base" in SAVED_MODELS else next(iter(SAVED_MODELS), None)

if DEMO_MODEL is None:
    print("Chưa có model để demo.")
else:
    demo_tokenizer, demo_model = load_saved_seq2seq(DEMO_MODEL)

    if DEMO_MODEL == "ViT5-base":
        demo_text = vietnamese_raw["test"][0]["article"]
        demo_prefix = ""
        demo_source_max = VI_SOURCE_MAX
        demo_new_tokens = VI_TARGET_MAX
    else:
        demo_text = english_raw["test"][0]["dialogue"]
        demo_prefix = EXPERIMENT_CONFIGS[DEMO_MODEL]["prefix"]
        demo_source_max = EN_SOURCE_MAX
        demo_new_tokens = EN_TARGET_MAX

    print("MODEL:", DEMO_MODEL)
    print("\nSOURCE:\n", demo_text[:1600])
    for strategy in ["greedy", "beam", "sampling"]:
        result = summarize_with_strategy(
            demo_text,
            demo_tokenizer,
            demo_model,
            prefix=demo_prefix,
            source_max=demo_source_max,
            strategy=strategy,
            max_new_tokens=demo_new_tokens,
        )
        print(f"\n{strategy.upper()}:\n{result}")

    del demo_model, demo_tokenizer
    clear_gpu()

<div class="exercise-box"><b>Nhận xét bắt buộc:</b> Chiến lược nào giữ đúng thông tin nhất? Sampling có thêm thông tin không có trong nguồn không? Beam search có bị dài hoặc lặp hơn greedy không?</div>

# PHẦN K — DEMO TÓM TẮT VĂN BẢN MỚI

In [ ]:
def summarize_saved_model(model_name, text, strategy="beam"):
    tokenizer, model = load_saved_seq2seq(model_name)

    if model_name == "T5-small":
        prefix, source_max, target_max = "summarize: ", EN_SOURCE_MAX, EN_TARGET_MAX
    elif model_name == "BART-base":
        prefix, source_max, target_max = "", EN_SOURCE_MAX, EN_TARGET_MAX
    elif model_name == "ViT5-base":
        prefix, source_max, target_max = "", VI_SOURCE_MAX, VI_TARGET_MAX
    else:
        prefix, source_max, target_max = "", 384, 96

    result = summarize_with_strategy(
        text=text,
        tokenizer=tokenizer,
        model=model,
        prefix=prefix,
        source_max=source_max,
        strategy=strategy,
        max_new_tokens=target_max,
    )
    del model, tokenizer
    clear_gpu()
    return result

In [ ]:
# Ví dụ tiếng Việt. Cell chỉ chạy khi ViT5 đã được huấn luyện.
new_vietnamese_text = """
Một nhóm sinh viên đã phát triển hệ thống hỗ trợ phân loại rác bằng trí tuệ nhân tạo.
Camera ghi nhận hình ảnh vật thể, mô hình nhận dạng loại rác và hướng dẫn người dùng bỏ vào đúng thùng.
Nhóm cho biết phiên bản thử nghiệm đạt độ chính xác cao trong môi trường phòng thí nghiệm,
nhưng hệ thống vẫn cần được kiểm thử thêm trong điều kiện ánh sáng và góc chụp đa dạng.
Dự án hướng tới giảm chi phí phân loại rác và nâng cao ý thức bảo vệ môi trường.
"""

if "ViT5-base" in SAVED_MODELS:
    print(summarize_saved_model("ViT5-base", new_vietnamese_text, strategy="beam"))
else:
    print("Hãy chạy phần ViT5 trước.")

## Demo Gradio tùy chọn

Bật `RUN_GRADIO = True` ở phần cấu hình rồi chạy cell dưới. Không bật mặc định để `Run all` không tự mở server.

In [ ]:
if RUN_GRADIO:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio>=5,<7"])
    import gradio as gr

    available_models = list(SAVED_MODELS)
    if not available_models:
        raise RuntimeError("Chưa có model đã huấn luyện.")

    def gradio_summarize(model_name, text, strategy):
        if not text.strip():
            return "Hãy nhập văn bản."
        return summarize_saved_model(model_name, text, strategy)

    demo = gr.Interface(
        fn=gradio_summarize,
        inputs=[
            gr.Dropdown(available_models, value=available_models[0], label="Mô hình"),
            gr.Textbox(lines=12, label="Văn bản nguồn"),
            gr.Radio(["greedy", "beam", "sampling"], value="beam", label="Giải mã"),
        ],
        outputs=gr.Textbox(lines=5, label="Bản tóm tắt"),
        title="LAB 09 — Transformer Text Summarization",
        description="Tóm tắt tiếng Anh bằng T5/BART hoặc tiếng Việt bằng ViT5.",
    )
    demo.launch(share=True, debug=False)
else:
    print("RUN_GRADIO=False — bỏ qua giao diện.")

# PHẦN L — LƯU KẾT QUẢ

In [ ]:
# Lưu bảng dự đoán để phân tích sau.
for model_name, df in PREDICTION_TABLES.items():
    filename = model_name.lower().replace("-", "_") + "_test_predictions.csv"
    path = OUTPUT_ROOT / filename
    df.to_csv(path, index=False)
    print("Đã lưu:", path)

print("\nCác thư mục/file trong output:")
for path in sorted(OUTPUT_ROOT.iterdir()):
    print("-", path)

In [ ]:
# Tùy chọn: nén toàn bộ kết quả để tải về máy.
archive_path = shutil.make_archive("/content/LAB09_models_and_results", "zip", OUTPUT_ROOT)
print("File ZIP:", archive_path)

# PHẦN M — CÂU HỎI THẢO LUẬN

1. Vì sao T5 cần tiền tố `summarize:` nhưng BART thường không cần?
2. Teacher forcing khác suy luận tự hồi quy như thế nào?
3. Vì sao ROUGE-2 thường thấp hơn ROUGE-1?
4. Truncation nguồn ảnh hưởng đến bản tóm tắt ra sao?
5. Tại sao sampling có thể nguy hiểm hơn beam search trong tóm tắt tin tức?
6. T5 và BART mô hình nào tốt hơn trên SAMSum của lần chạy này? Chênh lệch có đủ lớn để kết luận không?
7. Vì sao không nên dùng bảng chung để kết luận ViT5 mạnh hoặc yếu hơn T5/BART?
8. Hãy tìm ít nhất ba ví dụ mà ROUGE-L thấp nhưng bản tóm tắt vẫn chấp nhận được.
9. Hãy tìm ít nhất ba ví dụ có dấu hiệu hallucination.
10. Khi nào nên dùng mô hình extractive thay vì abstractive?

# PHẦN N — BÀI TẬP NỘP LAB

<div class="exercise-box">
<b>Yêu cầu tối thiểu</b>

1. Chạy đủ T5-small, BART-base và ViT5-base ở `QUICK_MODE=True`.
2. Nộp bảng benchmark và biểu đồ ROUGE.
3. Chọn 5 dự đoán tốt và 5 dự đoán yếu của mỗi ngôn ngữ.
4. So sánh greedy, beam và sampling trên ít nhất 3 văn bản.
5. Viết nhận xét 500–800 từ về chất lượng, tốc độ, VRAM và hallucination.
6. Tóm tắt 5 văn bản mới ngoài dataset.
</div>

### Bài nâng cao

- Tăng `QUICK_MODE=False` và so sánh lại.
- Thử `google/flan-t5-small` thay T5-small.
- Thử checkpoint `VietAI/vit5-base-vietnews-summarization` để suy luận trực tiếp rồi so với model tự fine-tune.
- Thêm BERTScore hoặc đánh giá thủ công theo ba tiêu chí: đầy đủ, mạch lạc, trung thực.
- Thử chunking cho bài báo dài vượt quá giới hạn token.
- Viết thuật toán map-reduce summarization cho nhiều đoạn.

# TỔNG KẾT

```text
Văn bản nguồn
    ↓ tokenizer + truncation
Encoder–Decoder Transformer
    ↓ fine-tuning bằng cặp nguồn–tóm tắt
Sinh tự hồi quy
    ↓ greedy / beam / sampling
Bản tóm tắt
    ↓
ROUGE + đánh giá thủ công + kiểm tra hallucination
```

Sau LAB 09, bước tiếp theo phù hợp là **LAB 10 — Question Answering và RAG trên tài liệu PDF**.

## Tài liệu tham khảo chính

- Hugging Face — Summarization task guide: https://huggingface.co/docs/transformers/tasks/summarization
- Hugging Face — T5 documentation: https://huggingface.co/docs/transformers/model_doc/t5
- Hugging Face — BART documentation: https://huggingface.co/docs/transformers/model_doc/bart
- T5-small model card: https://huggingface.co/google-t5/t5-small
- BART-base model card: https://huggingface.co/facebook/bart-base
- ViT5-base model card: https://huggingface.co/VietAI/vit5-base
- ViT5 VietNews checkpoint: https://huggingface.co/VietAI/vit5-base-vietnews-summarization
- SAMSum dataset card: https://huggingface.co/datasets/knkarthick/samsum
- VietNews dataset card: https://huggingface.co/datasets/ithieund/VietNews-Abs-Sum